## Settings - Download the data

In [ ]:
import os

from pathlib import Path
from dotenv import load_dotenv

import warnings
warnings.filterwarnings(action='ignore')

In [ ]:
# load_dotenv()
# os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLEUSERNAME")
# os.environ["KAGGLE_KEY"] = os.getenv("KAGGLEKEY")

In [ ]:
# from kaggle import api 

dataset_ref = "fedesoriano/heart-failure-prediction"
dest = Path.cwd().parent / "data"
# dest.mkdir(exist_ok=True)
target_file = dest / "heart.csv"

# if target_file.exists():
#     print(f"{target_file} already present, skipping download.")
# else:
#     api.dataset_download_files(dataset=dataset_ref, path=dest, unzip=True)
#     print(f"Downloaded dataset into {dest.resolve()}")

## Load the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(target_file)

In [ ]:
df.head().T.style.set_properties(**{'background-color': 'grey',
                           'color': 'white',
                           'border-color': 'white'})

In [ ]:
df.info()

## Explore the data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
def plot_histgram(feature):    
    fig = px.histogram(df, x=feature,
                       color="HeartDisease", 
                       marginal="box",
                       barmode ="overlay",
                       histnorm ='density'
                      )  
    fig.update_layout(
        title_font_color="white",
        legend_title_font_color="yellow",
        title={
            'text': feature+" histogram",
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top'},
        paper_bgcolor="black",
        plot_bgcolor='black',
        font_color="white"
    )
    fig.show()

In [ ]:
def display_stat(feature):
    mean = df[feature].mean()
    std = df[feature].std()
    skew = df[feature].skew()
    kurtosis = df[feature].kurtosis()
    print('mean: {0:.4f}, std: {1:.4f}, skew: {2:.4f}, kurtosis: {3:.4f} '.format(mean, std, skew, kurtosis))

In [ ]:
colors = ['gold', 'mediumturquoise']
labels = ['Normal','Heart Disease']
values = df['HeartDisease'].value_counts()/df['HeartDisease'].shape[0]

fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
fig.update_traces(hoverinfo='label+percent', textinfo='percent', textfont_size=20,
                  marker=dict(colors=colors, line=dict(color='#000000', width=2)))
fig.update_layout(
    title_text="Heart Disease",
    title_font_color="white",
    legend_title_font_color="yellow",
    paper_bgcolor="black",
    plot_bgcolor='black',
    font_color="white",
)
fig.show()

In [ ]:
plot_histgram('Age')
display_stat('Age')

In [ ]:
px.imshow(df.select_dtypes(include="number").corr(),title="Correlation Plot of the Heat Failure Prediction")

In [ ]:
pairplot_features = ["Age", "Cholesterol", "RestingBP", "MaxHR", "Oldpeak", "HeartDisease"]
subset = df[pairplot_features].sample(n=300, random_state=42)

sns.pairplot(subset, hue="HeartDisease", corner=True, plot_kws={"alpha":0.6, "s":30})
plt.suptitle("Targeted feature interactions", y=1.02)
plt.tight_layout()
plt.plot()

In [ ]:
plt.figure(figsize=(15,10))
for i,col in enumerate(df.columns,1):
    plt.subplot(4,3,i)
    plt.title(f"Distribution of {col} Data")
    sns.histplot(df[col],kde=True)
    plt.tight_layout()
    plt.plot()

### Data Quality Checks

We'll check for anomalies in the data before splitting into train/test sets. Note that we only **inspect** for issues here - all transformations will be done after the split to prevent data leakage.

In [ ]:
# Cholesterol and RestingBP should not be 0 - these likely represent missing values
zero_cholesterol = (df['Cholesterol'] == 0).sum()
zero_resting_bp = (df['RestingBP'] == 0).sum()

print(f"Records with Cholesterol = 0: {zero_cholesterol} ({zero_cholesterol/len(df)*100:.2f}%)")
print(f"Records with RestingBP = 0: {zero_resting_bp} ({zero_resting_bp/len(df)*100:.2f}%)")

if zero_cholesterol > 0 or zero_resting_bp > 0:
    print("\n⚠ Warning: Zero values detected - these may represent missing data")
    print("  Strategy: We'll handle these with median imputation in the ML pipeline")

In [ ]:
# Inspect for outliers (we'll retain them since tree-based models are robust to outliers)
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('HeartDisease')
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")

## ML Training

### Strategy to Prevent Data Leakage

1. **Split data FIRST** before any transformations
2. **All preprocessing in pipelines** - imputation, scaling, encoding happen within CV folds
3. **Nested Cross-Validation** - unbiased performance estimates with proper hyperparameter tuning
4. **Test set remains untouched** until final evaluation

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

### Step 1: Split Data (No Transformations Yet!)

We split the raw data first. The test set will be set aside and only used for final evaluation.

In [ ]:
# Separate features and target
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# Split FIRST - test set is now isolated
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nClass distribution in training set:\n{y_train.value_counts(normalize=True)}")

### Step 2: Build Preprocessing Pipelines

All data transformations are encapsulated in pipelines:
- **Zero imputation**: Treat zeros as missing values and impute with median
- **Scaling**: For linear models only
- **Encoding**: One-hot encode categorical variables

In [ ]:
# Identify feature types
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

# Custom function to replace zeros with NaN for specific columns
def replace_zeros_with_nan(X):
    X = X.copy()
    # Replace zeros with NaN for Cholesterol and RestingBP
    if 'Cholesterol' in X.columns:
        X.loc[X['Cholesterol'] == 0, 'Cholesterol'] = np.nan
    if 'RestingBP' in X.columns:
        X.loc[X['RestingBP'] == 0, 'RestingBP'] = np.nan
    return X

from sklearn.impute import SimpleImputer
# Preprocessing for numeric features (linear models): impute then scale
linear_numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
# Preprocessing for categorical features: impute (most frequent) then one-hot encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
linear_preprocessor = ColumnTransformer(
    transformers=[
        ('num', linear_numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)
# Preprocessing for tree-based models (no scaling needed): impute numeric, encode categorical
tree_numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', tree_numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)
# Wrapper to handle zero imputation
zero_imputer = FunctionTransformer(replace_zeros_with_nan, validate=False)

### Step 3: Create Complete Pipelines with Imputation

Each pipeline includes:
1. Zero-to-NaN conversion
2. Median imputation
3. Preprocessing (scaling/encoding)
4. Model

In [ ]:
# Random Forest Pipeline
rf_pipeline = Pipeline([
    ('zero_imputer', zero_imputer),
    ('preprocessor', tree_preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

# XGBoost Pipeline
xgb_pipeline = Pipeline([
    ('zero_imputer', zero_imputer),
    ('preprocessor', tree_preprocessor),
    ('model', XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1))
])

# Logistic Regression Pipeline
logreg_pipeline = Pipeline([
    ('zero_imputer', zero_imputer),
    ('preprocessor', linear_preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
])

### Step 4: Define Hyperparameter Search Spaces

Using RandomizedSearchCV instead of GridSearchCV for efficiency. We'll sample 50 random combinations for each model.

In [ ]:
# Random Forest hyperparameter distributions
rf_param_dist = {
    'model__n_estimators': randint(100, 1000),
    'model__max_depth': [None, 10, 20, 30, 40],
    'model__min_samples_split': randint(2, 20),
    'model__min_samples_leaf': randint(1, 10),
    'model__max_features': ['sqrt', 'log2', None],
    'model__bootstrap': [True, False]
}

# XGBoost hyperparameter distributions
xgb_param_dist = {
    'model__n_estimators': randint(100, 1000),
    'model__max_depth': randint(3, 10),
    'model__learning_rate': uniform(0.01, 0.29),  # 0.01 to 0.3
    'model__subsample': uniform(0.6, 0.4),  # 0.6 to 1.0
    'model__colsample_bytree': uniform(0.6, 0.4),  # 0.6 to 1.0
    'model__gamma': uniform(0, 1),  # 0 to 1
    'model__min_child_weight': randint(1, 10)
}

# Logistic Regression hyperparameter distributions
logreg_param_dist = {
    'model__C': uniform(0.01, 10),  # 0.01 to 10
    'model__penalty': ['l2'],
    'model__solver': ['liblinear', 'lbfgs', 'saga']
}

### Step 5: Nested Cross-Validation

**Nested CV Structure:**
- **Outer loop (5 folds)**: Provides unbiased performance estimates
- **Inner loop (3 folds)**: Hyperparameter tuning with RandomizedSearchCV

This prevents overfitting to the validation set and gives true generalization performance.

In [ ]:
# Cross-validation strategies
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'recall': 'recall',
    'precision': 'precision'
}

# Configure models with RandomizedSearchCV for hyperparameter tuning
models = {
    'Random Forest': RandomizedSearchCV(
        rf_pipeline,
        param_distributions=rf_param_dist,
        n_iter=50,
        cv=inner_cv,
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1,
        verbose=1
    ),
    'XGBoost': RandomizedSearchCV(
        xgb_pipeline,
        param_distributions=xgb_param_dist,
        n_iter=50,
        cv=inner_cv,
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1,
        verbose=1
    ),
    'Logistic Regression': RandomizedSearchCV(
        logreg_pipeline,
        param_distributions=logreg_param_dist,
        n_iter=20,
        cv=inner_cv,
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
}

### Step 6: Execute Nested Cross-Validation

This will take some time as we're doing:
- 5 outer folds × 50 inner iterations = 250 model fits for RF
- 5 outer folds × 50 inner iterations = 250 model fits for XGBoost
- 5 outer folds × 20 inner iterations = 100 model fits for LogReg

In [ ]:
# Store results
nested_cv_results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Running Nested CV for {name}")
    print(f"{'='*60}")
    
    # Outer loop: provides unbiased performance estimate
    cv_results = cross_validate(
        model, 
        X_train, 
        y_train,
        cv=outer_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    nested_cv_results[name] = cv_results
    
    print(f"\n{name} - Nested CV Results (Outer Loop):")
    print("-" * 60)
    for metric in ['accuracy', 'f1', 'roc_auc', 'recall', 'precision']:
        test_scores = cv_results[f'test_{metric}']
        print(f"{metric.upper():>15s}: {test_scores.mean():.4f} (±{test_scores.std():.4f})")

print("\n" + "="*60)
print("Nested CV Complete!")
print("="*60)

### Step 7: Train Final Models on Full Training Set

Now we train each model on the entire training set with hyperparameter tuning, then evaluate on the held-out test set.

In [ ]:
# Train each model on the full training set
final_models = {}

for name, model in models.items():
    print(f"\nTraining final {name} model...")
    model.fit(X_train, y_train)
    final_models[name] = model.best_estimator_
    
    print(f"Best hyperparameters for {name}:")
    for param, value in model.best_params_.items():
        print(f"  {param}: {value}")
    print(f"Best CV ROC-AUC: {model.best_score_:.4f}")

print("\nAll models trained successfully!")

## Model Evaluation and Interpretation

### Final Test Set Evaluation

**Important**: This is the first time we're using the test set. These scores represent the true generalization performance.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

print("="*80)
print("FINAL TEST SET EVALUATION")
print("="*80)

test_results = {}

for name, model in final_models.items():
    # Make predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    test_results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    
    print(f"\n{name}")
    print("-" * 80)
    print(f"Accuracy:  {test_results[name]['accuracy']:.4f}")
    print(f"Precision: {test_results[name]['precision']:.4f}")
    print(f"Recall:    {test_results[name]['recall']:.4f}")
    print(f"F1 Score:  {test_results[name]['f1']:.4f}")
    print(f"ROC-AUC:   {test_results[name]['roc_auc']:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

### Results Comparison: Nested CV vs Test Set

Comparing nested CV estimates with actual test performance helps verify our model selection process.

In [ ]:
# Create comparison dataframe
comparison_data = []

for name in final_models.keys():
    cv_roc_auc = nested_cv_results[name]['test_roc_auc'].mean()
    test_roc_auc = test_results[name]['roc_auc']
    
    comparison_data.append({
        'Model': name,
        'Nested CV ROC-AUC': cv_roc_auc,
        'Test ROC-AUC': test_roc_auc,
        'Difference': test_roc_auc - cv_roc_auc
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nNested CV vs Test Set Performance:")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison_df))
width = 0.35

ax.bar(x - width/2, comparison_df['Nested CV ROC-AUC'], width, label='Nested CV', alpha=0.8)
ax.bar(x + width/2, comparison_df['Test ROC-AUC'], width, label='Test Set', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('ROC-AUC Score')
ax.set_title('Nested CV vs Test Set Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ROC Curves

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(10, 8))

for name, model in final_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = test_results[name]['roc_auc']
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], '--', color='gray', linewidth=2, label='Random Classifier')
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves - Test Set", fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model) in enumerate(final_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                cbar_kws={'label': 'Count'})
    axes[idx].set_title(f"{name}\n(Accuracy: {test_results[name]['accuracy']:.3f})",
                       fontweight='bold')
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xticklabels(['No Disease', 'Disease'])
    axes[idx].set_yticklabels(['No Disease', 'Disease'])

plt.tight_layout()
plt.show()

### Feature Importance Analysis

Understanding which features contribute most to predictions.

In [ ]:
# Extract feature names after preprocessing
preprocessor = final_models['Random Forest'].named_steps['preprocessor']

# Get feature names from the ColumnTransformer
feature_names = []

# Numeric features
feature_names.extend(numeric_features)

# Categorical features (one-hot encoded)
cat_encoder = preprocessor.named_transformers_['cat']
if hasattr(cat_encoder, 'get_feature_names_out'):
    feature_names.extend(cat_encoder.get_feature_names_out(categorical_features))

# Plot feature importance for tree-based models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, model_name in enumerate(['Random Forest', 'XGBoost']):
    model = final_models[model_name]
    
    # Get feature importances
    importances = model.named_steps['model'].feature_importances_
    
    # Create dataframe and sort
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False).head(15)
    
    # Plot
    axes[idx].barh(range(len(importance_df)), importance_df['Importance'], alpha=0.8)
    axes[idx].set_yticks(range(len(importance_df)))
    axes[idx].set_yticklabels(importance_df['Feature'])
    axes[idx].set_xlabel('Importance')
    axes[idx].set_title(f'{model_name}\nTop 15 Features', fontweight='bold')
    axes[idx].invert_yaxis()
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

### Logistic Regression Coefficients

Examining the learned coefficients to understand feature effects.

In [ ]:
# Get logistic regression coefficients
logreg_model = final_models['Logistic Regression']
coefficients = logreg_model.named_steps['model'].coef_[0]

# Create dataframe
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values('Coefficient', key=abs, ascending=False).head(15)

# Plot
plt.figure(figsize=(10, 6))
colors = ['red' if c < 0 else 'green' for c in coef_df['Coefficient']]
plt.barh(range(len(coef_df)), coef_df['Coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(coef_df)), coef_df['Feature'])
plt.xlabel('Coefficient Value')
plt.title('Logistic Regression - Top 15 Feature Coefficients', fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3, axis='x')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nCoefficient Interpretation:")
print("- Positive coefficients increase the probability of heart disease")
print("- Negative coefficients decrease the probability of heart disease")

## Summary and Recommendations

### Key Findings

In [ ]:
# Find best model based on test Recall
best_model_name = max(test_results.items(), key=lambda x: x[1]['recall'])[0]
best_model_metrics = test_results[best_model_name]

print("="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\nBest Model: {best_model_name}")
print("-" * 80)
print(f"Test ROC-AUC:  {best_model_metrics['roc_auc']:.4f}")
print(f"Test Accuracy: {best_model_metrics['accuracy']:.4f}")
print(f"Test F1 Score: {best_model_metrics['f1']:.4f}")
print(f"Test Recall:   {best_model_metrics['recall']:.4f}")
print(f"Test Precision:{best_model_metrics['precision']:.4f}")

print("\n" + "="*80)
print("METHODOLOGY VALIDATION")
print("="*80)
print("✓ No data leakage - all transformations done within CV folds")
print("✓ Nested CV used - unbiased performance estimates")
print("✓ Test set untouched until final evaluation")
print("✓ RandomizedSearchCV used - efficient hyperparameter tuning")
print("\nThe model is ready for deployment with confidence in its generalization ability.")

## Save the best model for deployment

In [ ]:
import joblib

model_dir = Path.cwd().parent / "model"
model_dir.mkdir(exist_ok=True)

joblib.dump(final_models[best_model_name], model_dir / f"{best_model_name.replace(' ', '_').lower()}_best_model.pkl")

## SHAP Analysis

In [ ]:
from pathlib import Path
import numpy as np
import joblib

import shap
shap.initjs()

def replace_zeros_with_nan(X):
    X = X.copy()
    if 'Cholesterol' in X.columns:
        X.loc[X['Cholesterol'] == 0, 'Cholesterol'] = np.nan
    if 'RestingBP' in X.columns:
        X.loc[X['RestingBP'] == 0, 'RestingBP'] = np.nan
    return X


model_path = Path.cwd().parent / "model" / "random_forest_best_model.pkl"
print(f"Loading model from {model_path}")
loaded_model = joblib.load(model_path)
print("Model loaded successfully!")

In [ ]:
# 1. Apply zero_imputer
X_zero = loaded_model.named_steps["zero_imputer"].transform(X_test)

# 2. Apply ColumnTransformer
X_trans = loaded_model.named_steps["preprocessor"].transform(X_zero)


In [ ]:
feature_names = (
    loaded_model.named_steps["preprocessor"]
    .get_feature_names_out()
)
feature_names

In [ ]:
rf = loaded_model.named_steps["model"]
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_trans)

In [ ]:
X_trans.shape, shap_values.shape

In [ ]:
shap.summary_plot(
    shap_values[:, :, 1],
    X_trans,
    feature_names=feature_names
)


In [ ]:
feature_names

In [ ]:
# Generate a SHAP dependence plot for a specific feature (e.g., age)
shap.dependence_plot(
    "num__Age",
     shap_values[:, :, 1],
     X_trans,
     feature_names=feature_names
     )

In [ ]:
shap.force_plot(
    explainer.expected_value[1],   # class 1 expected value
    shap_values[:, :, 1],                       # SHAP values for sample i
    X_trans,                  # transformed features for sample i
    feature_names=feature_names,
    # matplotlib=True                 # render inline instead of JS
)


In [ ]:
i = 100
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[i, :, 1],
        base_values=explainer.expected_value[1],
        data=X_trans[i, :],
        feature_names=feature_names
    ),
    max_display=10
)
